In [3]:
from transformers import VisionEncoderDecoderModel, AutoConfig, GPT2Config

# 1. Load and modify Encoder Config for 8 frames
encoder_config = AutoConfig.from_pretrained("MCG-NJU/videomae-base")
encoder_config.num_frames = 8 # Tell the model to expect 8 frames

# 2. Load and modify Decoder Config (GPT-2)
decoder_config = GPT2Config.from_pretrained("gpt2")
decoder_config.is_decoder = True
decoder_config.add_cross_attention = True

# 3. Initialize the model with these explicit configs
model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(
    "MCG-NJU/videomae-base", 
    "gpt2", 
    encoder_config=encoder_config,
    decoder_config=decoder_config
)

# 4. Set special tokens and move to device
model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.to(device)

Loading weights: 100%|██████████| 184/184 [00:00<00:00, 2335.50it/s, Materializing param=layernorm.weight]                                 
VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     |  | 
---------------------------------------------------------------------+------------+--+-
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.bias            | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.weight           | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.bias          | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.value.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.v_bias       | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.bias      | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.key.weight   | UNEXPECTED |  | 
decoder.decode

VisionEncoderDecoderModel(
  (encoder): VideoMAEModel(
    (embeddings): VideoMAEEmbeddings(
      (patch_embeddings): VideoMAEPatchEmbeddings(
        (projection): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
    )
    (encoder): VideoMAEEncoder(
      (layer): ModuleList(
        (0-11): 12 x VideoMAELayer(
          (attention): VideoMAEAttention(
            (attention): VideoMAESelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
            )
            (output): VideoMAESelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): VideoMAEIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
          

In [6]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from transformers import VisionEncoderDecoderModel, AutoTokenizer, ViTImageProcessor, GPT2Config
from peft import LoraConfig, get_peft_model
from tqdm import tqdm

# 1. Setup Models (Using ViT instead of VideoMAE to avoid patch errors)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model_name = "google/vit-base-patch16-224-in21k"
decoder_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(decoder_name)
tokenizer.pad_token = tokenizer.eos_token
processor = ViTImageProcessor.from_pretrained(model_name)

# 2. Configure Decoder for Cross-Attention
decoder_config = GPT2Config.from_pretrained(decoder_name)
decoder_config.is_decoder = True
decoder_config.add_cross_attention = True

model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(
    model_name, decoder_name, decoder_config=decoder_config
)

# 3. Handle Token IDs and LoRA
model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id

for param in model.encoder.parameters():
    param.requires_grad = False

lora_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=["c_attn"], 
    lora_dropout=0.1, fan_in_fan_out=True
)
model.decoder = get_peft_model(model.decoder, lora_config)
model.to(device)

# 4. Updated Dataset for Frame-by-Frame Processing
class FrameDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
    
    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with np.load(row['processed_path']) as data:
            # ViT expects (Batch, C, H, W). We take the mean across the 8 frames
            video_array = data['video'] # (8, 3, 224, 224)
            pixel_values = torch.from_numpy(video_array).float().mean(dim=0) / 255.0
        
        labels = tokenizer(row['caption'], padding='max_length', max_length=30, 
                          truncation=True, return_tensors="pt").input_ids
        return {"pixel_values": pixel_values, "labels": labels.squeeze(0)}

# 5. Simple Training Loop
dataset = FrameDataset("msrvtt_2k_preprocessed.csv")
train_loader = DataLoader(dataset, batch_size=4, shuffle=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

for epoch in range(3):
    model.train()
    for batch in tqdm(train_loader):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(pixel_values=pixel_values, labels=labels)
        outputs.loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    print(f"Epoch {epoch+1} Complete")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2142.13it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                                                 | Status  | 
----------------------------------------------------+---------+-
transformer.h.{0...11}.crossattention.c_proj.bias   | MISSING | 
transformer.h.{0...11}.crossattention.c_attn.bias   | MISSING | 
transformer.h.{0...11}.ln_cross_attn.bias           | MISSING | 
transformer.h.{0...11}.crossattention.c_proj.weight | MISSING | 
transformer.h.{0...11}.crossattention.c_attn.weight | MISSING | 
transformer.h.{0...11}.crossattention.q_attn.weight | MISSING | 
transformer.h.{0...11}.crossattention.q_attn.bias   | MISSING | 
transformer.h.{0...11}.ln_cross_attn.weight         | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 10000/10000 [15:34<00:00, 10.70it

Epoch 1 Complete


100%|██████████| 10000/10000 [16:19<00:00, 10.21it/s]


Epoch 2 Complete


100%|██████████| 10000/10000 [29:01<00:00,  5.74it/s]   

Epoch 3 Complete


In [7]:
model.save_pretrained("./video_captioning_checkpoint_epoch3")
tokenizer.save_pretrained("./video_captioning_final")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.47it/s]


('./video_captioning_final/tokenizer_config.json',
 './video_captioning_final/tokenizer.json')

In [8]:
model.save_pretrained("./video_captioning_final")
tokenizer.save_pretrained("./video_captioning_final")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.79it/s]


('./video_captioning_final/tokenizer_config.json',
 './video_captioning_final/tokenizer.json')